# Notebook 00: Data-Quality Audit (Completed Solutions)

This notebook profiles the raw retail orders dataset (`data/raw/topic3-data.csv`) across all 10 structural issues identified in the workbook.

In [1]:
import sys
import re
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.utils import load_raw_data

df_raw = load_raw_data(PROJECT_ROOT / "data" / "raw" / "topic3-data.csv")
print(f"Raw dataset shape: {df_raw.shape} (480 rows x 15 columns)")
df_raw.head()

Raw dataset shape: (480, 15) (480 rows x 15 columns)


,order_id,order_date,ship_date,region,city,category,subcategory,sales_channel,customer_type,quantity,unit_price,discount,revenue,returned,notes
0,ORD-1001,2025-04-04,2025/04/11,Midwest,Chicago,Furniture,Chair,In-Store,Consumer,3,125.71,,377.12 USD,Y,.
1,ORD-1002,04-11-2025,2025-04-14,West,San Diego,Office Supplies,Paper,Online,Home Office,1,$14.04,15 percent,11.93,FALSE,priority
2,ORD-1002,4/11/25,Apr 14 2025,West,San Diego,Technology,Laptop,Online,Home Office,1,902.88,0.05,857.74 USD,No,
3,ORD-1002,4/11/25,2025/04/14,West,SanDiego,Office-Supplies,Paper,Online,Home Office,1 units,11.36,0,11,no,gift
4,ORD-1003,2/25/25,3/1/25,South,Dallas,Office Supplies,Binder,Online,Home Office,.,6.92,0.10,12.46 USD,Yes,


## 1. Exact Duplicate Rows
**Workbook target**: 16 rows are exact copies of another row.

In [2]:
dup_mask = df_raw.duplicated(keep="first")
print(f"Total exact duplicate rows: {dup_mask.sum()}")
df_raw[df_raw.duplicated(keep=False)].sort_values("order_id").head(10)

Total exact duplicate rows: 16


,order_id,order_date,ship_date,region,city,category,subcategory,sales_channel,customer_type,quantity,unit_price,discount,revenue,returned,notes
11,ORD-1006,3/31/25,2025/04/07,South,Nashville,Technology,Laptop,Online,Consumer,.,$931.30,none,"USD 1,862.59",FALSE,priority
473,ORD-1006,3/31/25,2025/04/07,South,Nashville,Technology,Laptop,Online,Consumer,.,$931.30,none,"USD 1,862.59",FALSE,priority
34,ORD-1020,May 23 2025,2025/05/28,Midwest,Madison,Furniture,Chair,In-Store,Consumer,1,116.51,0.10,$104.86,no,priority
466,ORD-1020,May 23 2025,2025/05/28,Midwest,Madison,Furniture,Chair,In-Store,Consumer,1,116.51,0.10,$104.86,no,priority
49,ORD-1028,2025-01-14,1/16/25,Midwest,Madison,Technology,Monitor,In-Store,Consumer,1,188.84,20%,$151.07,No,—
464,ORD-1028,2025-01-14,1/16/25,Midwest,Madison,Technology,Monitor,In-Store,Consumer,1,188.84,20%,$151.07,No,—
75,ORD-1044,2025-03-03,2025/03/10,midwest,Indianapolis,Office Supplies,Paper,Phone,Consumer,5 units,12.27,10%,55.22,FALSE,gift
477,ORD-1044,2025-03-03,2025/03/10,midwest,Indianapolis,Office Supplies,Paper,Phone,Consumer,5 units,12.27,10%,55.22,FALSE,gift
85,ORD-1048,04-06-2025,4/11/25,Midwest,Chicago,Office Supplies,Binder,Online,Corporate,1,USD 7.99,0%,$7.99,N,—
465,ORD-1048,04-06-2025,4/11/25,Midwest,Chicago,Office Supplies,Binder,Online,Corporate,1,USD 7.99,0%,$7.99,N,—


## 2. Repeated order_id & Table Grain
**Workbook target**: Only 260 unique order IDs across 480 rows.

In [3]:
order_counts = df_raw["order_id"].value_counts()
print(f"Unique order_ids: {df_raw['order_id'].nunique()}")
print(f"Orders with 1 line item:  {(order_counts == 1).sum()}")
print(f"Orders with >1 line items: {(order_counts > 1).sum()}")
order_counts.head(10)

Unique order_ids: 260
Orders with 1 line item:  103
Orders with >1 line items: 157


order_id
ORD-1044    4
ORD-1150    4
ORD-1228    4
ORD-1002    3
ORD-1004    3
ORD-1005    3
ORD-1017    3
ORD-1022    3
ORD-1024    3
ORD-1028    3
Name: count, dtype: int64

## 3. Date Formats & Unparseable Placeholders
**Workbook target**: 
- `order_date`: 4 valid formats mixed; 6 unparseable (impossible date '2025-02-30', 'not recorded').
- `ship_date`: 4 valid formats mixed; 4 'TBD' values.

In [4]:
# order_date profiling
order_parsed = pd.to_datetime(df_raw["order_date"], format="mixed", errors="coerce")
bad_order = df_raw.loc[order_parsed.isna(), "order_date"].value_counts()
print("Unparseable order_date values:")
print(bad_order)

# ship_date profiling
ship_parsed = pd.to_datetime(df_raw["ship_date"], format="mixed", errors="coerce")
bad_ship = df_raw.loc[ship_parsed.isna(), "ship_date"].value_counts()
print("\nUnparseable ship_date values:")
print(bad_ship)

Unparseable order_date values:
order_date
2025-02-30      3
not recorded    3
Name: count, dtype: int64

Unparseable ship_date values:
ship_date
TBD    4
Name: count, dtype: int64


## 4. Inconsistent Categorical Labels
**Workbook targets**:
- `region`: 21 distinct values for 4 regions
- `city`: 40 distinct values for 16 cities
- `category`: 17 distinct values for 3 categories
- `sales_channel`: 16 distinct values
- `customer_type`: 12 distinct values

In [5]:
cols = ["region", "city", "category", "sales_channel", "customer_type"]
for col in cols:
    unique_vals = sorted(df_raw[col].unique().tolist())
    print(f"\nColumn '{col}' ({len(unique_vals)} distinct values):")
    print(unique_vals[:10], "..." if len(unique_vals) > 10 else "")


Column 'region' (21 distinct values):
['', 'MW', 'Mid West', 'Midwest', 'Midwest ', 'NE', 'North East', 'Northeast', 'Northeast ', 'SOUTH'] ...

Column 'city' (40 distinct values):
['Atlanta', 'Atlanta ', 'Boston', 'Chicago', 'Chicago ', 'Dallas', 'Dallas ', 'Houston', 'Houston ', 'Indianapolis'] ...

Column 'category' (17 distinct values):
['', '.', 'Furniture', 'Furniture ', 'Furnitures', 'Office Supplies', 'Office Supplies ', 'Office-Supplies', 'Tech', 'Technology'] ...

Column 'sales_channel' (16 distinct values):
['E-Commerce', 'In Store', 'In store', 'In-Store', 'In-Store ', 'Online', 'Online ', 'PHONE', 'Phone', 'Phone '] ...

Column 'customer_type' (12 distinct values):
['Consumer', 'Consumer ', 'Corp', 'Corporate', 'Corporate ', 'Home Office', 'Home Office ', 'HomeOffice', 'consumer', 'corporate'] ...


## 5. Numbers Stored as Text
**Workbook targets**: Words ('two'), suffixes ('3 units'), currency symbols ('$', 'USD'), commas, padding.

In [6]:
print("Sample non-numeric quantities:")
print(df_raw.loc[~df_raw["quantity"].str.strip().str.isdigit(), "quantity"].value_counts())

print("\nSample unit_price formats:")
print(df_raw["unit_price"].head(10).tolist())

print("\nSample revenue formats:")
print(df_raw["revenue"].head(10).tolist())

Sample non-numeric quantities:
quantity
2 units    15
1 units    14
.          14
one         9
two         7
5 units     7
3 units     6
three       6
five        3
4 units     3
four        2
Name: count, dtype: int64

Sample unit_price formats:
['125.71', '$14.04', '902.88', '11.36', '6.92', '7.60', '81.07', '11.43', '3.47', '9.34']

Sample revenue formats:
['377.12 USD', '11.93', '857.74 USD', '11', '12.46 USD', '22.81 USD', '$81.07', 'unknown', '$6.24', '$15.87']


## 6. Negative and Bracketed Revenue
**Workbook target**: 23 rows written as -$37.30 or (103.75). Notice the correlation with the `returned` column!

In [7]:
neg_mask = df_raw["revenue"].str.contains(r"^-|^\(", regex=True, na=False)
neg_rows = df_raw[neg_mask]
print(f"Total negative/bracketed revenue rows: {len(neg_rows)}")
print("\nReturn status of negative revenue rows:")
print(neg_rows["returned"].value_counts())
neg_rows[["order_id", "quantity", "unit_price", "discount", "revenue", "returned"]].head(5)

Total negative/bracketed revenue rows: 23

Return status of negative revenue rows:
returned
TRUE    5
1       5
yes     5
Yes     5
Y       3
Name: count, dtype: int64


,order_id,quantity,unit_price,discount,revenue,returned
26,ORD-1016,1,69.08,10%,-$37.30,TRUE
55,ORD-1033,1,192,0.10,(103.75),1
154,ORD-1085,5,$3.28,0%,-$9.83,Y
203,ORD-1112,2,$227.18,5%,-$258.99,yes
219,ORD-1122,1 units,USD 7.85,15%,-$4.01,1


## 7. Discount Formats
**Workbook target**: 25 different spellings (0.10, .10, 10%, '10 percent', none, 'No discount', blank).

In [8]:
disc_counts = df_raw["discount"].value_counts(dropna=False)
print(f"Total unique discount representations: {len(disc_counts)}")
print(disc_counts)

Total unique discount representations: 25
discount
0%             57
0              56
10%            48
0.10           42
none           36
5%             31
0.05           30
0.15           19
15%            18
               17
No discount    15
.10            12
10 percent     12
0.20           12
5 percent      12
.05            12
 5%            10
20%             8
.15             7
 15%            6
15 percent      5
20 percent      5
.20             4
 20%            3
 10%            3
Name: count, dtype: int64


## 8. Return Flag Coding
**Workbook target**: 14 different codes for yes / no / unknown.

In [9]:
ret_counts = df_raw["returned"].value_counts(dropna=False)
print(f"Total unique return codes: {len(ret_counts)}")
print(ret_counts)

Total unique return codes: 14
returned
no         88
FALSE      83
No         77
0          73
N          64
1          17
yes        17
Y          14
Yes        14
TRUE       11
unknown     9
            6
—           5
.           2
Name: count, dtype: int64


## 9. Missing-Value Codes
**Workbook target**: blank, '.', em dash '—', 'unknown', 'TBD', 'not recorded', 'none'.

In [10]:
codes = ["", ".", "—", "unknown", "TBD", "not recorded", "none"]
missing_profile = []
for c in codes:
    total = (df_raw == c).sum().sum()
    missing_profile.append({"Code": repr(c), "Total Occurrences": total})
pd.DataFrame(missing_profile)

,Code,Total Occurrences
0,'',188
1,'.',68
2,'—',85
3,'unknown',41
4,'TBD',4
5,'not recorded',3
6,'none',36
